In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
from pathlib import Path

In [3]:
print(sys.path)

['/opt/homebrew/Cellar/python@3.11/3.11.8/Frameworks/Python.framework/Versions/3.11/lib/python311.zip', '/opt/homebrew/Cellar/python@3.11/3.11.8/Frameworks/Python.framework/Versions/3.11/lib/python3.11', '/opt/homebrew/Cellar/python@3.11/3.11.8/Frameworks/Python.framework/Versions/3.11/lib/python3.11/lib-dynload', '', '/Users/mike.sturwold/CodeProjects/ADM/dna-common-all-databricks-templates/Source/de_etl_examples/.venv/lib/python3.11/site-packages', '/Users/mike.sturwold/CodeProjects/ADM/dna-common-all-databricks-templates/Source/de_etl_examples/src', '/Users/mike.sturwold/CodeProjects/ADM/dna-common-all-databricks-templates/Source/de_etl_examples/.venv/lib/python3.11/site-packages/setuptools/_vendor', '/Users/mike.sturwold/CodeProjects/ADM/dna-common-all-databricks-templates/Source/adm_integration_framework']


In [4]:
root_path = (Path(os.getcwd()).parent.parent / "src").as_posix()
root_path

'/Users/mike.sturwold/CodeProjects/ADM/dna-common-all-databricks-templates/Source/adm_integration_framework/src'

In [5]:
sys.path.append(root_path)

In [14]:
import unittest
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, Row, DecimalType
from wilsonelser.transformation.data_transformer import DataTransformer
from wilsonelser.transformation.base_data_transformer import BaseDataTransformer
from wilsonelser.transformation.table_writer import TableWriter
from wilsonelser.transformation.table_reader import TableReader
import logging

In [15]:
config_file = "use_cases/p2c_insightops/sales_order_invoice_jdexeeu.yml"
table_reader = TableReader(config_file)
transformer = BaseDataTransformer(config_file)
transformer2 = DataTransformer(config_file)
writer = TableWriter(config_file)


In [ ]:
display(df.withColumn("y", F.lit("A")))

In [21]:
df = table_reader.read_source_table("jdexeeu_order_detail")
display(df.withColumn("x", 
                      F.to_date(F.substring(F.col("SDTRDJ").cast(StringType()),2,5), 'yyDDD')
                      )
         .withColumn("y", F.substring(F.col("SDTRDJ").cast(StringType()),2,1))
                      )

INFO:adm.integration.base_integration:Starting transformations for ID: jdexeeu_order_detail
INFO:adm.integration.base_integration:Applying function Name: use_cases.custom_transformations.p2c_tx.convert_julian_dates


,invoice_number,document_type,SDKCO,SDDOCO,SDLNID,sold_to_customer_no,entity_document_company_code,SDVR01,SDTORG,SDADDJ,SDTRDJ,SDIVD,user_id,x,y
0,4008353.0,RI,03009,4008231.0,1000.0,35013265.0,3009,,I068158,None,105014,105014,SQL-998010,2005-01-14,0
1,4008530.0,RI,03012,4008231.0,1000.0,35013737.0,3012,,I715062,None,105014,105014,SQL-998010,2005-01-14,0
2,4008530.0,RI,03012,4008231.0,2000.0,35013737.0,3012,,I715062,None,105014,105014,SQL-998010,2005-01-14,0
3,4009163.0,RI,03001,4008231.0,1000.0,35008876.0,3001,,I779063,None,105017,105017,SQL-998010,2005-01-17,0
4,4009163.0,RI,03001,4008231.0,2000.0,35008876.0,3001,,I779063,None,105017,105017,SQL-998010,2005-01-17,0
5,4007387.0,RI,03002,4008231.0,1000.0,35007305.0,3002,,I960776,None,104273,104273,SQL-998010,2004-09-29,0
6,4006793.0,RI,01793,4008231.0,1000.0,28016565.0,1793,2004.0.00591,I750512,2004-11-25,104306,104330,prod01793,2004-11-01,0
7,4008354.0,RI,03009,4008232.0,1000.0,35012848.0,3009,,I068158,None,105014,105014,SQL-998010,2005-01-14,0
8,4008531.0,RI,03012,4008232.0,1000.0,35014498.0,3012,,I715062,None,105014,105014,SQL-998010,2005-01-14,0
9,4008531.0,RI,03012,4008232.0,2000.0,35014498.0,3012,,I715062,None,105014,105014,SQL-998010,2005-01-14,0


In [12]:
df = transformer2.apply_joins("combine_1")

INFO:adm.integration.base_integration:Applying joins for combine_id: combine_1
INFO:adm.integration.base_integration:Starting transformations for ID: jdexeeu_order_detail
INFO:adm.integration.base_integration:Reading and selecting columns for source_id: jdexeeu_customer_ledger with alias: cl
INFO:adm.integration.base_integration:Starting transformations for ID: jdexeeu_customer_ledger
INFO:adm.integration.base_integration:Joining od with cl using condition: od.invoice_number = cl.RPDOC  and od.document_type = cl.RPDCT  and od.SDKCO = cl.RPKCO  and od.SDDOCO = cl.RPSDOC  and od.SDLNID = cl.RPLNID and join type: inner
INFO:adm.integration.base_integration:Schema after joining od with cl: StructType([StructField('invoice_number', DoubleType(), True), StructField('document_type', StringType(), True), StructField('sold_to_customer_no', DoubleType(), True), StructField('entity_document_company_code', IntegerType(), True), StructField('user_id', StringType(), True), StructField('SDKCO', Strin

In [13]:
display(df)
df.dtypes

,invoice_number,document_type,sold_to_customer_no,entity_document_company_code,user_id,SDKCO,SDDOCO,SDLNID,SDVR01,SDADDJ,SDADDJ_jd,RPDOC,RPDCT,RPKCO,RPSDOC,RPLNID,RPAG,RPACR,RPBCRC,payment_term_code,currency_code,payment_term_description,PNPTC,sold_to_customer_name,ABAN8,CCCO,COMPANY_CODE,entity_document_company_name,Region,segment,COUNTRY,CD_COPC,CD_LOCATION,CD_INTL_CREDIT_REP,LOCATION_CODE
0,4009348.0,RI,35013737.0,3012,SQL-998010,03012,4009065.0,1000.0,,0,None,4009348.0,RI,03012,4009065.0,1000.0,8400.0,0.0,XCD,002,XCD,CASH WITH ORDER,002,CASH CUSTOMER MISC - FEED,35013737.0,3012,3012.0,APLIGEN SA DE CV,SA,Nutrition,Mexico,3012.0,358,GBS,None
1,4009631.0,RI,35014415.0,3012,SQL-998010,03012,4009358.0,1000.0,,0,None,4009631.0,RI,03012,4009358.0,1000.0,8940.0,0.0,XCD,002,XCD,CASH WITH ORDER,002,KEITH JAPAL,35014415.0,3012,3012.0,APLIGEN SA DE CV,SA,Nutrition,Mexico,3012.0,358,GBS,None
2,4009662.0,RI,35014317.0,3012,SQL-998010,03012,4009388.0,1000.0,,0,None,4009662.0,RI,03012,4009388.0,1000.0,0.0,0.0,XCD,065,XCD,NO CHARGE,065,DONATIONS,35014317.0,3012,3012.0,APLIGEN SA DE CV,SA,Nutrition,Mexico,3012.0,358,GBS,None
3,4005990.0,RI,28375898.0,1793,prod01793,01793,4008382.0,1000.0,2041648/2041650/2041649,104314,None,4005990.0,RI,01793,4008382.0,1000.0,1880200.0,0.0,EUR,007,EUR,30 DAYS FROM INVOICE DATE,007,TWILMIJ B V,28375898.0,1793,1793.0,ADM Specialty Ingredients (Europe) B.V.,EMEA,Nutrition,Netherlands,NaN,None,None,None
4,4006411.0,RI,35019890.0,1793,PROD01793,01793,4008873.0,1000.0,pcn 918,104322,None,4006411.0,RI,01793,4008873.0,1000.0,2185500.0,0.0,EUR,009,EUR,60 DAYS FROM INVOICE DATE,009,PROVIMI LTD,35019890.0,1793,1793.0,ADM Specialty Ingredients (Europe) B.V.,EMEA,Nutrition,Netherlands,NaN,None,None,None
5,4006932.0,RI,28655297.0,1793,prod01793,01793,4009145.0,1000.0,2120-12,104330,None,4006932.0,RI,01793,4009145.0,1000.0,1818571.0,0.0,EUR,007,EUR,30 DAYS FROM INVOICE DATE,007,PHOSPHOLIPID GMBH,28655297.0,1793,1793.0,ADM Specialty Ingredients (Europe) B.V.,EMEA,Nutrition,Netherlands,NaN,None,None,None
6,4006753.0,RI,35018826.0,1793,prod01793,01793,4009483.0,1000.0,2042578,104331,None,4006753.0,RI,01793,4009483.0,1000.0,1899430.0,0.0,EUR,007,EUR,30 DAYS FROM INVOICE DATE,007,Verveka Veevoeder BV,35018826.0,1793,1793.0,ADM Specialty Ingredients (Europe) B.V.,EMEA,Nutrition,Netherlands,NaN,None,None,None
7,4009305.0,RI,35008380.0,3001,SQL-998010,03001,4008345.0,1000.0,,0,None,4009305.0,RI,03001,4008345.0,1000.0,40800000.0,0.0,JMD,002,JMD,CASH WITH ORDER,002,"BAYLEY, GEORGE",35008380.0,3001,NaN,None,None,None,None,3001.0,311,GBS,None
8,4007488.0,RI,35013341.0,3002,SQL-998010,03002,4008347.0,1000.0,,0,None,4007488.0,RI,03002,4008347.0,1000.0,761080.0,0.0,JMD,003,JMD,CASH ON DELIVERY,003,MORETON PARK SUPERMARKET,35013341.0,3002,NaN,None,None,None,None,3002.0,316,GBS,None
9,4009367.0,RI,35008902.0,3001,SQL-998010,03001,4008399.0,1000.0,,0,None,4009367.0,RI,03001,4008399.0,1000.0,29595000.0,0.0,JMD,003,JMD,CASH ON DELIVERY,003,SUEPAT'S RICE & SUGAR SALES,35008902.0,3001,NaN,None,None,None,None,3001.0,311,GBS,None


[('invoice_number', 'double'),
 ('document_type', 'string'),
 ('sold_to_customer_no', 'double'),
 ('entity_document_company_code', 'int'),
 ('user_id', 'string'),
 ('SDKCO', 'string'),
 ('SDDOCO', 'double'),
 ('SDLNID', 'double'),
 ('SDVR01', 'string'),
 ('SDADDJ', 'decimal(18,0)'),
 ('SDADDJ_jd', 'date'),
 ('RPDOC', 'double'),
 ('RPDCT', 'string'),
 ('RPKCO', 'string'),
 ('RPSDOC', 'double'),
 ('RPLNID', 'double'),
 ('RPAG', 'double'),
 ('RPACR', 'double'),
 ('RPBCRC', 'string'),
 ('payment_term_code', 'string'),
 ('currency_code', 'string'),
 ('payment_term_description', 'string'),
 ('PNPTC', 'string'),
 ('sold_to_customer_name', 'string'),
 ('ABAN8', 'double'),
 ('CCCO', 'int'),
 ('COMPANY_CODE', 'int'),
 ('entity_document_company_name', 'string'),
 ('Region', 'string'),
 ('segment', 'string'),
 ('COUNTRY', 'string'),
 ('CD_COPC', 'int'),
 ('CD_LOCATION', 'string'),
 ('CD_INTL_CREDIT_REP', 'string'),
 ('LOCATION_CODE', 'string')]

In [113]:
df2 = transformer.apply_transformations("jdexeeu_joined", df)

INFO:adm.integration.base_integration:Starting transformations for ID: jdexeeu_joined


In [114]:
display(df2)

,invoice_number,total_invoice_amount_usd,payment_term_code,entity_document_company_code,entity_document_company_name,user_id,payment_term_description,sold_to_customer_no,document_type,sold_to_customer_no,sold_to_customer_name,source_system,currency_code,region,document_type,segment,country,gbs_group,__inserted_on
0,4009348,84.00,002,3012,APLIGEN SA DE CV,SQL-998010,CASH WITH ORDER,35013737,RI,35013737,CASH CUSTOMER MISC - FEED,jdeEU,XCD,SA,RI,Nutrition,Mexico,Y,2025-04-28
1,4009631,89.40,002,3012,APLIGEN SA DE CV,SQL-998010,CASH WITH ORDER,35014415,RI,35014415,KEITH JAPAL,jdeEU,XCD,SA,RI,Nutrition,Mexico,Y,2025-04-28
2,4009662,0.00,065,3012,APLIGEN SA DE CV,SQL-998010,NO CHARGE,35014317,RI,35014317,DONATIONS,jdeEU,XCD,SA,RI,Nutrition,Mexico,Y,2025-04-28
3,4005990,18802.00,007,1793,ADM Specialty Ingredients (Europe) B.V.,prod01793,30 DAYS FROM INVOICE DATE,28375898,RI,28375898,TWILMIJ B V,jdeEU,EUR,EMEA,RI,Nutrition,Netherlands,N,2025-04-28
4,4006411,21855.00,009,1793,ADM Specialty Ingredients (Europe) B.V.,PROD01793,60 DAYS FROM INVOICE DATE,35019890,RI,35019890,PROVIMI LTD,jdeEU,EUR,EMEA,RI,Nutrition,Netherlands,N,2025-04-28
5,4006932,18185.71,007,1793,ADM Specialty Ingredients (Europe) B.V.,prod01793,30 DAYS FROM INVOICE DATE,28655297,RI,28655297,PHOSPHOLIPID GMBH,jdeEU,EUR,EMEA,RI,Nutrition,Netherlands,N,2025-04-28
6,4006753,18994.30,007,1793,ADM Specialty Ingredients (Europe) B.V.,prod01793,30 DAYS FROM INVOICE DATE,35018826,RI,35018826,Verveka Veevoeder BV,jdeEU,EUR,EMEA,RI,Nutrition,Netherlands,N,2025-04-28
7,4009305,408000.00,002,3001,None,SQL-998010,CASH WITH ORDER,35008380,RI,35008380,"BAYLEY, GEORGE",jdeEU,JMD,None,RI,None,None,Y,2025-04-28
8,4007488,7610.80,003,3002,None,SQL-998010,CASH ON DELIVERY,35013341,RI,35013341,MORETON PARK SUPERMARKET,jdeEU,JMD,None,RI,None,None,Y,2025-04-28
9,4009367,295950.00,003,3001,None,SQL-998010,CASH ON DELIVERY,35008902,RI,35008902,SUEPAT'S RICE & SUGAR SALES,jdeEU,JMD,None,RI,None,None,Y,2025-04-28


In [104]:
df = table_reader.read_source_table("jdexeeu_order_detail")
df.columns

INFO:adm.integration.base_integration:Starting transformations for ID: jdexeeu_order_detail


['invoice_number',
 'document_type',
 'SDKCO',
 'SDDOCO',
 'SDLNID',
 'sold_to_customer_no',
 'entity_document_company_code',
 'SDVR01',
 'SDTORG',
 'SDADDJ',
 'SDTRDJ',
 'SDIVD',
 'user_id']

In [76]:
df = table_reader.read_source_table("jdexeeu_customer_ledger")
df.columns

INFO:adm.integration.base_integration:Starting transformations for ID: jdexeeu_customer_ledger


['RPDOC',
 'RPDCT',
 'RPKCO',
 'RPSDOC',
 'RPLNID',
 'payment_term_code',
 'currency_code',
 'RPBCRC',
 'RPAG',
 'RPACR',
 'RPDDJ']

In [89]:
df = table_reader.read_source_table("jdexeeu_payment_terms")
df.columns

INFO:adm.integration.base_integration:Starting transformations for ID: jdexeeu_payment_terms


['PNPTC', 'payment_term_description']

In [78]:
df = table_reader.read_source_table("jdexeeu_foundation_env")
df.columns

INFO:adm.integration.base_integration:Starting transformations for ID: jdexeeu_foundation_env


['CCCO']

In [79]:
df = table_reader.read_source_table("jdexeeu_currencytable")
df.columns

['MANDT', 'KURST', 'FCURR', 'TCURR', 'GDATU', 'UKURS', 'FFACT', 'TFACT']

In [80]:
df = table_reader.read_source_table("jdexeeu_address_book_master")
df.columns

INFO:adm.integration.base_integration:Starting transformations for ID: jdexeeu_address_book_master


['ABAN8', 'sol_to_customer_name']

In [57]:
df = table_reader.read_source_table("jdexeeu_entity_mapping")
df.dtypes

INFO:adm.integration.base_integration:Starting transformations for ID: jdexeeu_entity_mapping


[('COMPANY_CODE', 'string'),
 ('entity_document_company_name', 'string'),
 ('Region', 'string'),
 ('segment', 'string'),
 ('COUNTRY', 'string')]

In [58]:
display(df)

,COMPANY_CODE,entity_document_company_name,Region,segment,COUNTRY
0,2294,ADM GUATEMALA LIMITADA,SA,Ag Services & Oilseeds,Guatemala
1,3210,ADM AGRO INDUSTRIES KOTA & AKOLA PVT. LTD.,APAC,Ag Services & Oilseeds,India
2,4821,ADM AUSTRALIA LOGISTICS PTY LTD,APAC,Ag Services & Oilseeds,Australia
3,675,ADM AUSTRALIA LOGISTICS PTY LTD,APAC,Ag Services & Oilseeds,Australia
4,4894,ADM AUSTRALIA LOGISTICS PTY LTD,APAC,Ag Services & Oilseeds,Australia
5,3826,OLENEX EDIBLE OILS GMBH,EMEA,Ag Services & Oilseeds,Germany
6,70718,*Unknown*,SA,Corporate,Brazil
7,1241,MALTA-TEXO DE MEXICO SA DE CV,SA,Nutrition,Mexico
8,2725,CI ADM COLOMBIA LTDA.,SA,Ag Services & Oilseeds,Colombia
9,2898,EVIALIS INDIA LIMITED,APAC,Nutrition,India


In [84]:
# df = table_reader.read_source_table("jdexeeu_copc_cross_reference")
# df.columns

In [86]:
# df = table_reader.read_source_table("jdexeeu_copc_profit_center")
# df.columns

In [87]:
df = table_reader.read_source_table("jdexeeu_credit_dist_list")
df.columns

INFO:adm.integration.base_integration:Starting transformations for ID: jdexeeu_credit_dist_list


['CD_COPC', 'CD_LOCATION', 'CD_INTL_CREDIT_REP']

In [88]:
df = table_reader.read_source_table("jdexeeu_gbs_emea_billing_entities")
df.columns

INFO:adm.integration.base_integration:Starting transformations for ID: jdexeeu_gbs_emea_billing_entities


['LOCATION_CODE']

In [93]:
df = transformer2.apply_joins("combine_1")

INFO:adm.integration.base_integration:Applying joins for combine_id: combine_1
INFO:adm.integration.base_integration:Starting transformations for ID: jdexeeu_order_detail
INFO:adm.integration.base_integration:Starting transformations for ID: jdexeeu_customer_ledger


INFO:adm.integration.base_integration:Columns in cl: ['RPDOC', 'RPDCT', 'RPKCO', 'RPSDOC', 'RPLNID', 'payment_term_code', 'currency_code', 'RPBCRC', 'RPAG', 'RPACR', 'RPDDJ']
INFO:adm.integration.base_integration:Duplicate column detected and skipped: rr.requisition_id


AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `ttf`.`requisition_id` cannot be resolved. Did you mean one of the following? [`od`.`user_id`, `od`.`document_type`, `cl`.`RPACR`, `cl`.`RPAG`, `cl`.`RPBCRC`]. SQLSTATE: 42703;
'Project ['ttf.requisition_id AS requisition_id#45564, 'ttf.job_type AS job_type#45565, 'ttf.pay_grade_band AS pay_grade_band#45566, 'ttf.hiring_manager_name AS hiring_manager_name#45567, 'ttf.recruiter_name AS recruiter_name#45568, 'ttf.organizational_level_2 AS organizational_level_2#45569, 'ttf.organizational_level_3 AS organizational_level_3#45570, 'ttf.business_id AS business_id#45571, 'rr.division AS division#45572, 'rr.old_reason_for_hire AS old_reason_for_hire#45573, 'rr.ec_reason_for_hire AS ec_reason_for_hire#45574, 'rr.additional_replacement_justification AS additional_replacement_justification#45575, 'rr.college_relations_requisition AS college_relations_requisition#45576, 'rr.fte_number_of_posi

In [73]:
df = transformer.apply_transformations("ffjt", df)

INFO:adm.integration.base_integration:Starting transformations for ID: ffjt


In [ ]:
writer.write_table("silver_globlal_hr_request_poc", df)

INFO:adm.integration.base_integration:Getting schema from schema file use_cases/h2r/h2r_silver_head_count_poc_schema.json for sandbox.de_ref_silver_mikesturwoldcontractor.silver_head_count_poc


INFO:adm.integration.base_integration:Number of records written to table sandbox.de_ref_silver_mikesturwoldcontractor.silver_head_count_poc: 200


In [41]:
df = table_reader.read_source_table("req_report")
#df_payroll = table_reader.read_source_table("payroll")
#df_employee = table_reader.read_source_table("employee")
#df_employee = transformer.apply_transformations("employee_new", df_employee)
#df_employee = transformer.apply_transformations("employee_expr", df_employee)

#df = transformer2.apply_joins("emp_payroll")
# df = transformer.apply_transformations("employee", df)
#df = transformer.apply_transformations("employee_expr_new", df_employee)
#df = df.select(tgt_cols)
#writer = TableWriter("scratch/mike/h2r_silver_head_count_poc.yml")
#writer.write_table("silver_head_count_poc", df)

INFO:adm.integration.base_integration:Starting transformations for ID: req_report


In [0]:
#Source/adm_integration_framework/scratch/mike/h2r_head_count_poc.yml
#table_reader = TableReader("scratch/mike/h2r_silver_head_count_poc.yml")
#df_payroll = table_reader.read_source_table("payroll")
#df_employee= table_reader.read_source_table("employee")
#display(df_employee.select("bus_id","bus_id_trim"))
#display(result_df.selectExpr("* except(Emp_ID, First_Name)"))

In [0]:
#transformer = BaseDataTransformer("scratch/mike/h2r_head_count_poc.yml")
#tx = "rn"

#result_df = transformer.apply_transformations("rn", result_df)
#result_df = transformer.apply_transformations("cast_cols", result_df)
#display(result_df)
#display(transformed_df.selectExpr("*", "cast(overtime_pay as decimal(10,2)) as overtime_pay"))
# why do i need to setup logger here?
# fixed by adding self.logger to adm_base_data_transformer

# column expressions duplicate columns if * specified

#transformer = DataTransformer("scratch/mike/h2r_silver_head_count_poc.yml")
#result_df = transformer.apply_joins("emp_payroll")




In [0]:
display(result_df)

In [0]:
df = result_df.transform(p2c_custom_tranformatin, Col, cole3)

In [0]:
writer = TableWriter("scratch/mike/h2r_silver_head_count_poc.yml")
# why does writer create the table?
# why does it need a schema file?
writer.write_table("silver_head_count_poc", result_df)

In [0]:
for x in spark.table("sandbox.h2r_dbx_poc.ec_employee").columns:
    print(x)

In [0]:
for x in spark.table("sandbox.h2r_dbx_poc.ec_payroll").columns:
    print(x)